# Rate Limiting with RedisRateLimitGuardrail

This notebook demonstrates `RedisRateLimitGuardrail`, a Redis-backed rate limiting guardrail for the OpenAI Agents SDK.

## Features

- **Per-user rate limiting** - Track and enforce limits for individual users
- **Request count limits** - Cap the number of requests per time window
- **Token usage limits** - Cap total tokens consumed per time window
- **Sliding and fixed window algorithms** - Choose the strategy that fits your use case
- **SDK-compatible guardrail interface** - Plug directly into OpenAI Agents SDK `InputGuardrail`

## Use Cases

- Prevent abuse by capping requests per user
- Control costs by limiting token consumption
- Protect downstream services from bursts of traffic
- Implement fair-use policies across multi-tenant agents

In [ ]:
import os
import asyncio

from redis_openai_agents import RedisRateLimitGuardrail
from redis_openai_agents.rate_limit_guardrail import GuardrailFunctionOutput

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

print(f"Using Redis at {REDIS_URL}")

## 1. Basic Rate Limiting (Per-User, Request-Based)

The simplest usage: limit each user to a fixed number of requests per time window.
We use a short 10-second window here for demonstration purposes.

In [ ]:
# Create a rate limiter: 3 requests per 10-second window
basic_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_basic",
    requests_per_minute=3,
    window_seconds=10,
)
await basic_limiter.initialize()

user_id = "user_alice"

print(f"Rate limit: 3 requests per 10 seconds for user '{user_id}'")
print("=" * 60)

for i in range(5):
    result = await basic_limiter.check_rate_limit(user_id=user_id)
    status = "BLOCKED" if result.tripwire_triggered else "ALLOWED"
    print(f"  Request {i + 1}: {status}")
    if result.tripwire_triggered:
        print(f"    Reason: {result.output_info['reason']}")

await basic_limiter.close()

## 2. Sliding vs Fixed Window Strategies

The guardrail supports two windowing algorithms:

- **Sliding window** (`window_type="sliding"`, the default): Uses a Redis sorted set with timestamps. Old entries are continuously pruned, giving a smooth rolling window.
- **Fixed window** (`window_type="fixed"`): Uses a simple counter that resets at fixed intervals. Simpler but can allow bursts at window boundaries.

Below we compare both strategies side by side.

In [ ]:
# Sliding window rate limiter
sliding_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_sliding",
    requests_per_minute=3,
    window_type="sliding",
    window_seconds=10,
)
await sliding_limiter.initialize()

# Fixed window rate limiter
fixed_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_fixed",
    requests_per_minute=3,
    window_type="fixed",
    window_seconds=10,
)
await fixed_limiter.initialize()

user_id = "user_bob"

print("Comparing sliding vs fixed window (3 requests / 10s):")
print("=" * 60)

for i in range(5):
    sliding_result = await sliding_limiter.check_rate_limit(user_id=user_id)
    fixed_result = await fixed_limiter.check_rate_limit(user_id=user_id)

    sliding_status = "BLOCKED" if sliding_result.tripwire_triggered else "ALLOWED"
    fixed_status = "BLOCKED" if fixed_result.tripwire_triggered else "ALLOWED"

    print(f"  Request {i + 1}:  Sliding={sliding_status}  |  Fixed={fixed_status}")

# Wait for the window to expire, then show that requests are allowed again
print("\nWaiting 11 seconds for window to expire...")
await asyncio.sleep(11)

sliding_result = await sliding_limiter.check_rate_limit(user_id=user_id)
fixed_result = await fixed_limiter.check_rate_limit(user_id=user_id)

sliding_status = "BLOCKED" if sliding_result.tripwire_triggered else "ALLOWED"
fixed_status = "BLOCKED" if fixed_result.tripwire_triggered else "ALLOWED"
print(f"  After reset: Sliding={sliding_status}  |  Fixed={fixed_status}")

await sliding_limiter.close()
await fixed_limiter.close()

## 3. Token-Based Rate Limiting

Instead of (or in addition to) counting requests, you can limit based on **token usage**.
Pass `tokens_used` to `check_rate_limit()` to track cumulative token consumption.

In [ ]:
# Token-only rate limiter: 500 tokens per 10-second window
token_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_tokens",
    tokens_per_minute=500,
    window_type="sliding",
    window_seconds=10,
)
await token_limiter.initialize()

user_id = "user_carol"

# Simulate requests with varying token counts
token_requests = [150, 200, 100, 100]

print(f"Token limit: 500 tokens per 10 seconds for user '{user_id}'")
print("=" * 60)

cumulative = 0
for i, tokens in enumerate(token_requests):
    result = await token_limiter.check_rate_limit(
        user_id=user_id,
        tokens_used=tokens,
    )
    if not result.tripwire_triggered:
        cumulative += tokens
    status = "BLOCKED" if result.tripwire_triggered else "ALLOWED"
    print(f"  Request {i + 1} ({tokens} tokens): {status}  [cumulative: {cumulative}/500]")
    if result.tripwire_triggered:
        print(f"    Reason: {result.output_info['reason']}")

await token_limiter.close()

## 4. Combined Request + Token Limits

You can enforce **both** request count and token limits simultaneously.
A request is blocked if **either** limit is exceeded.

In [ ]:
# Combined limiter: 5 requests AND 300 tokens per 10-second window
combined_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_combined",
    requests_per_minute=5,
    tokens_per_minute=300,
    window_type="sliding",
    window_seconds=10,
)
await combined_limiter.initialize()

user_id = "user_dave"

# Each request uses 100 tokens -- token limit (300) will trigger before
# request limit (5)
print("Combined limits: 5 requests AND 300 tokens per 10 seconds")
print("Each request uses 100 tokens")
print("=" * 60)

for i in range(6):
    result = await combined_limiter.check_rate_limit(
        user_id=user_id,
        tokens_used=100,
    )
    status = "BLOCKED" if result.tripwire_triggered else "ALLOWED"
    print(f"  Request {i + 1} (100 tokens): {status}")
    if result.tripwire_triggered:
        print(f"    Reason: {result.output_info['reason']}")

await combined_limiter.close()

## 5. Getting Rate Limit Info

Use `get_rate_limit_info()` to inspect a user's current usage and remaining quota.

In [ ]:
info_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_info",
    requests_per_minute=10,
    tokens_per_minute=1000,
    window_type="sliding",
    window_seconds=60,
)
await info_limiter.initialize()

user_id = "user_eve"

# Make a few requests to build up usage
for _ in range(3):
    await info_limiter.check_rate_limit(user_id=user_id, tokens_used=150)

# Query the current rate limit status
info = await info_limiter.get_rate_limit_info(user_id=user_id)

print(f"Rate limit info for user '{user_id}':")
print("=" * 60)
print(f"  Requests:  {info['current_requests']}/{info['requests_limit']}")
print(f"  Remaining: {info['remaining_requests']} requests")
print(f"  Tokens:    {info['current_tokens']}/{info['tokens_limit']}")
print(f"  Remaining: {info['remaining_tokens']} tokens")

await info_limiter.close()

## 6. Integration with OpenAI Agents SDK

The `RedisRateLimitGuardrail` provides a `guardrail_function` method that is compatible
with the OpenAI Agents SDK `InputGuardrail`. It returns a `GuardrailFunctionOutput`
with `tripwire_triggered=True` when the rate limit is exceeded.

The guardrail function extracts the `user_id` from the run context automatically:

- If `context.context` has a `user_id` attribute, it uses that
- If `context.context` is a dict with a `"user_id"` key, it uses that
- Otherwise it falls back to `"default"`

Below is how you would wire it up with an agent.

In [ ]:
# Demonstrate the guardrail_function interface
#
# In a real application you would use:
#
#   from agents import Agent, InputGuardrail
#
#   rate_limiter = RedisRateLimitGuardrail(
#       redis_url=REDIS_URL,
#       requests_per_minute=60,
#       tokens_per_minute=10000,
#   )
#   await rate_limiter.initialize()
#
#   guardrail = InputGuardrail(
#       guardrail_function=rate_limiter.guardrail_function,
#       name="rate_limit",
#   )
#   agent = Agent(
#       name="MyAgent",
#       instructions="You are a helpful assistant.",
#       input_guardrails=[guardrail],
#   )
#
# Here we call guardrail_function directly with mock objects to show
# the GuardrailFunctionOutput that the SDK would receive.

from dataclasses import dataclass


@dataclass
class MockUserContext:
    user_id: str


@dataclass
class MockRunContext:
    context: MockUserContext


sdk_limiter = RedisRateLimitGuardrail(
    redis_url=REDIS_URL,
    key_prefix="demo_sdk",
    requests_per_minute=2,
    window_seconds=10,
)
await sdk_limiter.initialize()

ctx = MockRunContext(context=MockUserContext(user_id="user_frank"))

print("Calling guardrail_function (as the SDK would):")
print("=" * 60)

for i in range(4):
    output: GuardrailFunctionOutput = await sdk_limiter.guardrail_function(
        context=ctx,
        agent=None,  # Not used by the guardrail
        input_data="Hello, agent!",
    )
    status = "TRIPWIRE TRIGGERED" if output.tripwire_triggered else "PASSED"
    print(f"  Request {i + 1}: {status}")
    print(f"    output_info: {output.output_info}")

# Verify it is a proper GuardrailFunctionOutput
print(f"\nReturn type: {type(output).__name__}")
print(f"Has tripwire_triggered: {hasattr(output, 'tripwire_triggered')}")
print(f"Has output_info: {hasattr(output, 'output_info')}")

await sdk_limiter.close()

## Summary

`RedisRateLimitGuardrail` provides a complete rate limiting solution backed by Redis:

| Feature | Description |
|---|---|
| **Per-user tracking** | Each user has independent rate limit counters |
| **Request limits** | Cap number of requests via `requests_per_minute` |
| **Token limits** | Cap token consumption via `tokens_per_minute` |
| **Combined limits** | Enforce both request and token limits simultaneously |
| **Sliding window** | Smooth rolling window using Redis sorted sets |
| **Fixed window** | Simple counter-based window with automatic reset |
| **Introspection** | `get_rate_limit_info()` returns current usage and remaining quota |
| **SDK integration** | `guardrail_function` works directly with `InputGuardrail` |

### Key Methods

- `initialize()` / `close()` - Manage the Redis connection
- `check_rate_limit(user_id, tokens_used=0)` - Check and record a request
- `get_rate_limit_info(user_id)` - Query current usage without consuming quota
- `guardrail_function(context, agent, input_data)` - SDK-compatible guardrail entry point

In [ ]:
# Cleanup: flush the demo keys
import redis.asyncio as aioredis

client = aioredis.from_url(REDIS_URL, decode_responses=True)

# Remove all demo keys created in this notebook
prefixes = [
    "demo_basic:*",
    "demo_sliding:*",
    "demo_fixed:*",
    "demo_tokens:*",
    "demo_combined:*",
    "demo_info:*",
    "demo_sdk:*",
]

total_deleted = 0
for pattern in prefixes:
    cursor = None
    while cursor != 0:
        cursor, keys = await client.scan(cursor=cursor or 0, match=pattern, count=100)
        if keys:
            await client.delete(*keys)
            total_deleted += len(keys)

await client.aclose()
print(f"Cleanup complete. Deleted {total_deleted} demo keys.")